In [ ]:
"""
KoChatGPT 데이터셋 EDA 워크북
========================================
【 대상 데이터 】
- kochatgpt_1_SFT.jsonl (SFT 학습용)

【 분석 체크리스트 】
1. 도메인 분석 (이 데이터가 뭘 다루는 건가?)
2. 문체 분석 (존댓말/반말/격식체 등)
3. 길이 분포 (극단적인 것들이 있는가?)
4. 문장 완성도 (끝이 뭔가 이상한 건 없는가?)
5. 특수 패턴 (따옴표, 이모티콘, 특문 등)
6. 통계 요약 (평균, 중앙값, 편차 등)
"""

In [3]:
import json
import re
from collections import Counter
import statistics


# 1단계: 데이터 로드

In [5]:
# === Colab 호환 셋업 (자동 추가) ===
import torch as _t
_t._orig_load = getattr(_t, '_orig_load', _t.load)
def _compat_load(*a, **k):
    k.setdefault('weights_only', False)
    return _t._orig_load(*a, **k)
_t.load = _compat_load
try:
    import matplotlib; matplotlib.rcParams['axes.unicode_minus'] = False
except Exception: pass
print('[colab-compat] torch.load weights_only=False 패치')


[colab-compat] torch.load weights_only=False 패치


In [6]:
!git clone https://github.com/airobotlab/KoChatGPT
!cp -r /content/KoChatGPT/colossalai_ChatGPT_230319/chatgpt /content/chatgpt

Cloning into 'KoChatGPT'...
remote: Enumerating objects: 304, done.
remote: Total 304 (delta 0), reused 0 (delta 0), pack-reused 304 (from 1)
Receiving objects: 100% (304/304), 57.72 MiB | 19.68 MiB/s, done.
Resolving deltas: 100% (123/123), done.


In [7]:
import os

modifications = [
    {
        "file": "/content/chatgpt/trainer/callbacks/save_checkpoint.py",
        "changes": [
            {"line": 3, "old": "from chatgpt.trainer.strategies import ColossalAIStrategy, Strategy",
             "new": "from chatgpt.trainer.strategies import Strategy"},
            {"line": 71, "old": "only_rank0 = not isinstance(self.strategy, ColossalAIStrategy)",
             "new": "            only_rank0 = not isinstance(self.strategy)"},
        ],
    },
    {
        "file": "/content/chatgpt/trainer/strategies/__init__.py",
        "changes": [
            {"line": 1, "old": "from .colossalai import ColossalAIStrategy", "new": ""},  # 삭제
            {"line": 5, "old": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy', 'ColossalAIStrategy']",
             "new": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy']"},
        ],
    },
    {
        "file": "/content/chatgpt/dataset/reward_dataset.py",
        "changes": [
            {"line": 3, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ],
    },
    {
        "file": "/content/chatgpt/trainer/base.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    },
    {
        "file": "/content/chatgpt/trainer/rm.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    }
]


def modify_file(file_path, changes):
    """파일에서 지정된 줄을 찾아 내용을 수정하는 함수"""

    if not os.path.exists(file_path):
        print(f"⚠️ 파일이 존재하지 않습니다: {file_path}")
        return

    with open(file_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    modified = False

    for change in changes:
        line_index = change["line"]
        if 0 <= line_index < len(lines):
            if lines[line_index].strip() == change["old"]:
                lines[line_index] = change["new"] + "\n"
                modified = True
            else:
                print(f"⚠️ {file_path} 파일의 {change['line']}번째 줄이 예상과 다릅니다.")
                print(f"   예상: {change['old']}")
                print(f"   실제: {lines[line_index].strip()}")

    if modified:
        with open(file_path, "w", encoding="utf-8") as file:
            file.writelines(lines)
        print(f"✅ 수정 완료: {file_path}")
    else:
        print(f"⚠️ {file_path} 수정할 내용이 없습니다.")

for mod in modifications:
    modify_file(mod["file"], mod["changes"])

✅ 수정 완료: /content/chatgpt/trainer/callbacks/save_checkpoint.py
✅ 수정 완료: /content/chatgpt/trainer/strategies/__init__.py
✅ 수정 완료: /content/chatgpt/dataset/reward_dataset.py
✅ 수정 완료: /content/chatgpt/trainer/base.py
✅ 수정 완료: /content/chatgpt/trainer/rm.py


In [8]:

print("=" * 80)
print("1단계: 데이터 로드")
print("=" * 80)

SFT_FILE = '/content/KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl'

try:
    with open(SFT_FILE, 'r', encoding='utf-8') as f:
        sft_data = json.load(f)
    print(f"✓ {SFT_FILE} 로드 성공")
    print(f"  총 {len(sft_data)}건의 레코드\n")
except FileNotFoundError:
    print(f"✗ 파일을 찾을 수 없음: {SFT_FILE}")
    sft_data = []

# 첫 몇 개 샘플 확인
print("【 첫 3개 샘플 확인 】")
for i, record in enumerate(sft_data[:3]):
    print(f"\n샘플 {i+1}:")
    print(f"  prompt ({len(record['prompt'])}자): {record['prompt']}")
    print(f"  completion ({len(record['completion'])}자): {record['completion'][:80]}...")

1단계: 데이터 로드
✓ /content/KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl 로드 성공
  총 12000건의 레코드

【 첫 3개 샘플 확인 】

샘플 1:
  prompt (13자): 불고기용 고기 한우에요?
  completion (203자): '저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 ...

샘플 2:
  prompt (14자): 쓰던 앱이 유료로 전환됐어
  completion (305자): '어떤 앱인지 모르기 때문에 정확한 답변을 드리기 어렵습니다. 하지만, 일반적으로 유료 전환된 앱은 기존 무료 앱에서 추가적인 기능이나 서비스를...

샘플 3:
  prompt (7자): 여친이랑 다툼
  completion (160자): '저는 인공지능이기 때문에 여친과의 다툼에 대해서는 조언을 드릴 수는 없습니다. 하지만 다투게 되면 상대방의 말을 경청하는 태도와 서로의 감정을...


# 2단계: 기본 통계

In [9]:

print("\n" + "=" * 80)
print("2단계: 기본 통계")
print("=" * 80)

prompts = [r['prompt'] for r in sft_data]
completions = [r['completion'] for r in sft_data]
tokens = [r.get('tokens', 0) for r in sft_data if 'tokens' in r]

print(f"\n【 레코드 수 】")
print(f"  총 레코드: {len(sft_data)}건")
print(f"  유니크 prompt: {len(set(prompts))}건 ({len(set(prompts))/len(prompts)*100:.1f}%)")
print(f"  유니크 completion: {len(set(completions))}건 ({len(set(completions))/len(completions)*100:.1f}%)")

print(f"\n【 Prompt 길이 분포 (자)】")
p_lens = [len(p) for p in prompts]
print(f"  최소: {min(p_lens)}, 최대: {max(p_lens)}")
print(f"  평균: {statistics.mean(p_lens):.1f}, 중앙값: {statistics.median(p_lens):.1f}")
print(f"  표준편차: {statistics.stdev(p_lens):.1f}")

print(f"\n【 Completion 길이 분포 (자)】")
c_lens = [len(c) for c in completions]
print(f"  최소: {min(c_lens)}, 최대: {max(c_lens)}")
print(f"  평균: {statistics.mean(c_lens):.1f}, 중앙값: {statistics.median(c_lens):.1f}")
print(f"  표준편차: {statistics.stdev(c_lens):.1f}")

if tokens:
    print(f"\n【 Token 수 분포 】")
    print(f"  최소: {min(tokens)}, 최대: {max(tokens)}")
    print(f"  평균: {statistics.mean(tokens):.1f}, 중앙값: {statistics.median(tokens):.1f}")


2단계: 기본 통계

【 레코드 수 】
  총 레코드: 12000건
  유니크 prompt: 11946건 (99.6%)
  유니크 completion: 11974건 (99.8%)

【 Prompt 길이 분포 (자)】
  최소: 0, 최대: 295
  평균: 22.2, 중앙값: 19.0
  표준편차: 14.1

【 Completion 길이 분포 (자)】
  최소: 4, 최대: 1553
  평균: 144.1, 중앙값: 118.0
  표준편차: 122.8

【 Token 수 분포 】
  최소: 17, 최대: 1111
  평균: 159.3, 중앙값: 134.0


# 3단계: 특이점 탐지

In [10]:
print("\n" + "=" * 80)
print("3단계: 특이점 탐지")
print("=" * 80)

print(f"\n【 빈 또는 매우 짧은 데이터 】")
empty_p = sum(1 for p in prompts if not p.strip())
empty_c = sum(1 for c in completions if not c.strip())
short_p = sum(1 for p in prompts if len(p.strip()) < 5)
short_c = sum(1 for c in completions if len(c.strip()) < 10)
print(f"  빈 prompt: {empty_p}건")
print(f"  빈 completion: {empty_c}건")
print(f"  극단적 단답 prompt (<5자): {short_p}건")
print(f"  극단적 단답 completion (<10자): {short_c}건")

print(f"\n【 중복 】")
dup_p = len(prompts) - len(set(prompts))
dup_c = len(completions) - len(set(completions))
print(f"  중복 prompt: {dup_p}건")
print(f"  중복 completion: {dup_c}건")

# 가장 많이 반복되는 prompt/completion
from collections import Counter
p_counts = Counter(prompts)
c_counts = Counter(completions)
print(f"\n【 가장 많이 반복되는 prompt (상위 5) 】")
for p, cnt in p_counts.most_common(5):
    print(f"  '{p[:30]}...' → {cnt}회")

print(f"\n【 가장 많이 반복되는 completion (상위 5) 】")
for c, cnt in c_counts.most_common(5):
    print(f"  '{c[:30]}...' → {cnt}회")


3단계: 특이점 탐지

【 빈 또는 매우 짧은 데이터 】
  빈 prompt: 3건
  빈 completion: 0건
  극단적 단답 prompt (<5자): 125건
  극단적 단답 completion (<10자): 115건

【 중복 】
  중복 prompt: 54건
  중복 completion: 26건

【 가장 많이 반복되는 prompt (상위 5) 】
  '얼마에요?...' → 5회
  '이거는 얼마예요?...' → 4회
  '영수증 좀 주세요...' → 4회
  '...' → 3회
  '화장실은 어디 있어요?...' → 3회

【 가장 많이 반복되는 completion (상위 5) 】
  ''2018년입니다....' → 4회
  ''문재인 대통령입니다....' → 4회
  ''미국입니다....' → 4회
  ''2019년입니다....' → 3회
  ''1953년입니다....' → 2회


## 3-1. 극단적 단답 데이터 상세 분석

In [11]:
# 극단적 단답 Prompt (< 5자) 상위 5개
short_prompts = [(idx, p) for idx, p in enumerate(prompts) if len(p.strip()) < 5]

print(f"=== 극단적 단답 Prompt 샘플 (총 {len(short_prompts)}건 중 상위 5개) ===")
for i, (idx, p) in enumerate(short_prompts[:5], 1):
    print(f"{i}. [Index {idx}] '{p}' (길이: {len(p)}자)")
    print(f"   연관 Completion: {completions[idx][:60]}...")
    print("-" * 50)

print("\n" + "="*60 + "\n")

# 극단적 단답 Completion (< 10자) 상위 5개
short_completions = [(idx, c) for idx, c in enumerate(completions) if len(c.strip()) < 10]

print(f"=== 극단적 단답 Completion 샘플 (총 {len(short_completions)}건 중 상위 5개) ===")
for i, (idx, c) in enumerate(short_completions[:5], 1):
    print(f"{i}. [Index {idx}] '{c}' (길이: {len(c)}자)")
    print(f"   연관 Prompt: {prompts[idx]}")
    print("-" * 50)

=== 극단적 단답 Prompt 샘플 (총 125건 중 상위 5개) ===
1. [Index 116] '헤어짐' (길이: 3자)
   연관 Completion: '헤어짐은 두 사람 사이의 관계가 끝나는 것을 의미합니다. 이는 종종 감정적인 상처와 슬픔을 초래할 수 있습...
--------------------------------------------------
2. [Index 300] '먼들' (길이: 2자)
   연관 Completion: '저는 인공지능 언어 모델로서 먼들이라는 단어에 대한 정확한 의미 파악이 어렵습니다. 먼들이 어떤 문맥에서 ...
--------------------------------------------------
3. [Index 324] '기다림' (길이: 3자)
   연관 Completion: '기다림은 기다리는 것을 의미합니다. 즉, 어떤 일, 사건, 인물 등이 일어날 때까지 기다리는 것입니다. 기...
--------------------------------------------------
4. [Index 391] '과식했다' (길이: 4자)
   연관 Completion: '앗, 그럴 땐 조금 쉬어가면서 소화를 돕는 음식을 섭취하거나 건강한 습관으로 조금씩 먹도록 노력해보세요. ...
--------------------------------------------------
5. [Index 415] '휴우.' (길이: 3자)
   연관 Completion: '안녕하세요. 무슨 문제가 있으신가요? 제가 도와드릴게요....
--------------------------------------------------


=== 극단적 단답 Completion 샘플 (총 115건 중 상위 5개) ===
1. [Index 17] ''한국입니다.' (길이: 7자)
   연관 Prompt: 2000년 5월 허위사실유포죄를 위헌판정한 나라는?
-----------------------------------------------

## 3-2. 가장 많이 중복된 데이터 상세 분석

In [12]:
from collections import Counter

# 중복 Prompt 상위 5개와 해당 인덱스들
p_counts = Counter(prompts)
top_dup_prompts = [p for p, cnt in p_counts.most_common(5) if cnt > 1]

print("=== 가장 많이 중복되는 Prompt 상위 5개 ===")
for i, p_text in enumerate(top_dup_prompts, 1):
    cnt = p_counts[p_text]
    # 해당 Prompt가 등장하는 모든 인덱스 찾기
    indices = [idx for idx, p in enumerate(prompts) if p == p_text]
    print(f"{i}. [반복 {cnt}회] Prompt: '{p_text}'")
    print(f"   등장 인덱스: {indices}")
    print("-" * 50)

print("\n" + "="*60 + "\n")

# 중복 Completion 상위 5개와 해당 인덱스들
c_counts = Counter(completions)
top_dup_completions = [c for c, cnt in c_counts.most_common(5) if cnt > 1]

print("=== 가장 많이 중복되는 Completion 상위 5개 ===")
for i, c_text in enumerate(top_dup_completions, 1):
    cnt = c_counts[c_text]
    # 해당 Completion이 등장하는 모든 인덱스 찾기
    indices = [idx for idx, c in enumerate(completions) if c == c_text]
    print(f"{i}. [반복 {cnt}회] Completion: '{c_text[:60]}...'")
    print(f"   등장 인덱스: {indices}")
    print("-" * 50)

=== 가장 많이 중복되는 Prompt 상위 5개 ===
1. [반복 5회] Prompt: '얼마에요?'
   등장 인덱스: [1571, 4061, 7627, 8760, 11091]
--------------------------------------------------
2. [반복 4회] Prompt: '이거는 얼마예요?'
   등장 인덱스: [1201, 1755, 5564, 8995]
--------------------------------------------------
3. [반복 4회] Prompt: '영수증 좀 주세요'
   등장 인덱스: [2203, 4582, 7882, 11928]
--------------------------------------------------
4. [반복 3회] Prompt: ''
   등장 인덱스: [1983, 4793, 8222]
--------------------------------------------------
5. [반복 3회] Prompt: '화장실은 어디 있어요?'
   등장 인덱스: [2104, 10855, 11785]
--------------------------------------------------


=== 가장 많이 중복되는 Completion 상위 5개 ===
1. [반복 4회] Completion: ''2018년입니다....'
   등장 인덱스: [2123, 6386, 6764, 11896]
--------------------------------------------------
2. [반복 4회] Completion: ''문재인 대통령입니다....'
   등장 인덱스: [2263, 9599, 10935, 11783]
--------------------------------------------------
3. [반복 4회] Completion: ''미국입니다....'
   등장 인덱스: [2680, 3802, 6627, 7777]
-----------------------

# 4단계: 문체 분석

In [13]:
print("\n" + "=" * 80)
print("4단계: 문체 분석")
print("=" * 80)

# 종결어미 분석 (따옴표 제거 후 분석, 상호배타적)
print(f"\n【 Completion 종결어미 패턴 】")

def strip_quotes_for_analysis(text):
    """분석용 따옴표 제거 (맨 앞뒤만)"""
    text = text.strip()
    text = text.lstrip("'\"")
    text = text.rstrip("'\"")
    return text.strip()

completions_cleaned = [strip_quotes_for_analysis(c) for c in completions]

# 상호배타적으로 분류 (카테고리별로 한 번만 카운트)
from collections import Counter
ending_categories = []
for c in completions_cleaned:
    if not c:
        ending_categories.append('빈문장')
    elif c.endswith('?'):
        ending_categories.append('의문문 (?)')
    elif c.endswith('!'):
        ending_categories.append('감탄문 (!)')
    elif c.endswith('.'):
        ending_categories.append('마침표 (.)')
    else:
        ending_categories.append('기타 (이상한 끝)')

counter = Counter(ending_categories)

# 0%인 카테고리는 표시하지 않음
for category in ['마침표 (.)', '감탄문 (!)', '의문문 (?)', '기타 (이상한 끝)', '빈문장']:
    count = counter.get(category, 0)
    if count > 0:
        print(f"  {category}: {count}건 ({count/len(completions)*100:.1f}%)")

# 특수문자 분석
print(f"\n【 특수문자 사용 현황 】")

# 1. 따옴표: 원본에서 세기 (정제 전 데이터의 특성)
quotes_single = sum(1 for c in completions if "'" in c)
quotes_double = sum(1 for c in completions if '"' in c)

print(f"  따옴표 포함 (원본):")
print(f"    - 작은따옴표(') 포함: {quotes_single}건 ({quotes_single/len(completions)*100:.1f}%)")
print(f"    - 큰따옴표(\") 포함: {quotes_double}건 ({quotes_double/len(completions)*100:.1f}%)")

# 2. 이모티콘: 정제된 텍스트에서 세기
emoji_count = sum(1 for c in completions_cleaned if any(ord(ch) > 0xFFFF for ch in c))
print(f"\n  이모티콘/특수 유니코드: {emoji_count}건 ({emoji_count/len(completions)*100:.1f}%)")

# 3. 파싱 오류 지표 (dict 구조가 텍스트로 섞인 경우)
has_token = sum(1 for c in completions if 'token' in c)
print(f"\n  파싱 오류 지표 ('token' 포함): {has_token}건 ({has_token/len(completions)*100:.1f}%)")

# 4. 이스케이프 문자
has_escape = sum(1 for c in completions if '\\n' in c or '\\\\' in c)
print(f"  이스케이프 문자 (\\n, \\\\): {has_escape}건 ({has_escape/len(completions)*100:.1f}%)")


4단계: 문체 분석

【 Completion 종결어미 패턴 】
  마침표 (.): 10670건 (88.9%)
  감탄문 (!): 427건 (3.6%)
  의문문 (?): 187건 (1.6%)
  기타 (이상한 끝): 716건 (6.0%)

【 특수문자 사용 현황 】
  따옴표 포함 (원본):
    - 작은따옴표(') 포함: 12000건 (100.0%)
    - 큰따옴표(") 포함: 1446건 (12.0%)

  이모티콘/특수 유니코드: 11건 (0.1%)

  파싱 오류 지표 ('token' 포함): 505건 (4.2%)
  이스케이프 문자 (\n, \\): 1238건 (10.3%)


## 4-1 : 이상한 끝 (716건) 패턴 상세 분석

In [14]:
"""
마침표, !, ? 이외의 종결 패턴을 분석하고
각 패턴을 카테고리별로 분류
"""

import json
import re
from collections import Counter

# ============================================================================
# 데이터 로드
# ============================================================================

completions = [r['completion'] for r in sft_data]

# 따옴표 제거
def strip_quotes(text):
    text = text.strip()
    text = text.lstrip("'\"")
    text = text.rstrip("'\"")
    return text.strip()

completions_cleaned = [strip_quotes(c) for c in completions]

# ============================================================================
# 이상한 끝 추출
# ============================================================================

weird_endings = []
for idx, c in enumerate(completions_cleaned):
    if c and not c.endswith(('.', '!', '?')):
        weird_endings.append((idx, c))

print(f"【 이상한 끝 기본 통계 】\n")
print(f"총 {len(weird_endings)}건의 '이상한 끝' 발견")
print(f"비율: {len(weird_endings)/len(completions)*100:.1f}%\n")

# ============================================================================
# 패턴 분류
# ============================================================================

print("="*80)
print("패턴 분류 중...\n")

categories = {
    'dict_parsing_error': [],      # ", 'token': 숫자}" 패턴
    'escape_char': [],              # \n, \\, \t 등
    'english_mixed': [],            # 영문이나 혼합 응답
    'emoji': [],                    # 이모티콘이나 :) 등으로 끝남
    'incomplete': [],               # 불완전한 문장
}

# 패턴 정의
dict_error_pattern = re.compile(r"['\"]\s*,\s*['\"]token['\"]:|}\s*$")
escape_pattern = re.compile(r'\\[nrt\\]|\\x[0-9a-f]{2}')
english_pattern = re.compile(r'^[a-zA-Z]|[a-zA-Z]\s*$|[A-Z]{2,}$')
emoji_pattern = re.compile(r'[:;=]-?[\)\(\]D]|😊|🙃|😍|👍|💯|✓|☑|※')

for idx, c in weird_endings:
    tail = c[-50:]  # 마지막 50자

    # 1. dict 파싱 오류
    if 'token' in c and ('}' in c[-10:] or "'" in c[-10:]):
        categories['dict_parsing_error'].append((idx, c))
    # 2. 이스케이프 문자
    elif escape_pattern.search(c[-30:]):
        categories['escape_char'].append((idx, c))
    # 3. 이모티콘
    elif emoji_pattern.search(c[-20:]):
        categories['emoji'].append((idx, c))
    # 4. 영문이나 혼합
    elif english_pattern.search(c) or any(ord(ch) < 128 for ch in c[-30:]):
        categories['english_mixed'].append((idx, c))
    # 5. 불완전한 문장
    else:
        categories['incomplete'].append((idx, c))

# ============================================================================
# 결과 출력 - 테이블
# ============================================================================

print("="*80)
print("【 이상한 끝 (716건) 패턴 분석 】\n")

# 테이블 헤더
header = f"{'패턴':<20} | {'건수':>6} | {'비율':>8}"
print(header)
print("-" * 80)

# 데이터
table_data = [
    ("dict 파싱 실패", len(categories['dict_parsing_error'])),
    ("이스케이프 문자", len(categories['escape_char'])),
    ("영문/혼합 응답", len(categories['english_mixed'])),
    ("이모티콘", len(categories['emoji'])),
    ("불완전 문장", len(categories['incomplete'])),
]

total = len(weird_endings)
for pattern, count in table_data:
    ratio = count / total * 100
    print(f"{pattern:<20} | {count:>6} | {ratio:>7.1f}%")

print("-" * 80)
print(f"{'합계':<20} | {total:>6} | {100.0:>7.1f}% |")
print()

# ============================================================================
# 상세 분석 - 카테고리별 샘플
# ============================================================================

print("="*80)
print("【 카테고리별 상세 분석 】\n")

for category_name, category_list in categories.items():
    if not category_list:
        continue

    category_display = {
        'dict_parsing_error': 'dict 파싱 실패',
        'escape_char': '이스케이프 문자',
        'english_mixed': '영문/혼합 응답',
        'emoji': '이모티콘',
        'incomplete': '불완전 문장',
    }

    print(f"\n【 {category_display[category_name]} 】")
    print(f"총 {len(category_list)}건\n")

    # 처음 5개 샘플
    for i, (idx, c) in enumerate(category_list[:5], 1):
        print(f"{i}. [Index {idx}]")

        # 'dict_parsing_error' 카테고리인 경우에만 마지막 15자 포함하여 출력
        if category_name == 'dict_parsing_error':
            print(f"   내용: {c[-100:]}")
            print(f"   마지막 15자: {repr(c[-15:])}")
        else:
            # 그 외 카테고리는 내용만 출력 (기존처럼 뒤 100자만 보려면 {c[-100:]}, 전체 보려면 {c})
            print(f"   내용: {c}")


【 이상한 끝 기본 통계 】

총 716건의 '이상한 끝' 발견
비율: 6.0%

패턴 분류 중...

【 이상한 끝 (716건) 패턴 분석 】

패턴                   |     건수 |       비율
--------------------------------------------------------------------------------
dict 파싱 실패           |    505 |    70.5%
이스케이프 문자             |     31 |     4.3%
영문/혼합 응답             |    114 |    15.9%
이모티콘                 |     52 |     7.3%
불완전 문장               |     14 |     2.0%
--------------------------------------------------------------------------------
합계                   |    716 |   100.0% |

【 카테고리별 상세 분석 】


【 dict 파싱 실패 】
총 505건

1. [Index 15]
   내용: 어느 나라의 의원을 역임했는지, 어떤 의회의 의원이었는지 등의 정보가 필요합니다. 이에 대한 정보를 제공해 주시면 더 정확한 답변을 드릴 수 있습니다.", 'token': 154}
   마지막 15자: ", 'token': 154}"
2. [Index 57]
   내용: 한명숙 장관이 리더쉽 평가에서 1위로 선정된 신문은 '조선일보'입니다.", 'token': 85}
   마지막 15자: '", \'token\': 85}'
3. [Index 106]
   내용: 최남현을 스타덤으로 오르게 한 연극 작품은 '모두의 여왕'입니다.", 'token': 79}
   마지막 15자: '", \'token\': 79}'
4. [Index 155]
   내용: 표된 '통일기본법'을 바탕으로 1992년 3월 12일 국회에서 통과

# 5단계: 도메인 분석

In [18]:

print("\n" + "=" * 80)
print("5단계: 도메인 분석 (샘플 기반)")
print("=" * 80)

# 임의의 10개 샘플을 보고 도메인 파악
print(f"\n【 10개 랜덤 샘플로 도메인 파악 】")
print("(아래의 샘플들을 읽고 어떤 주제들이 있는지 파악하세요)\n")

import random
random.seed(42)
sample_indices = random.sample(range(len(sft_data)), min(10, len(sft_data)))

for idx in sample_indices[:10]:
    r = sft_data[idx]
    print(f"Q: {r['prompt']}")
    print(f"A: {r['completion'][:100]}...")
    print()



5단계: 도메인 분석 (샘플 기반)

【 10개 랜덤 샘플로 도메인 파악 】
(아래의 샘플들을 읽고 어떤 주제들이 있는지 파악하세요)

Q: 막스와 미샤엘이 설립한 공장 중에 이후 모든 제작을 맡게 된 곳은?
A: "막스와 미샤엘이 설립한 공장 중 이후 모든 제작을 맡게 된 곳은 'Adidas(아디다스)'입니다.", 'token': 91}...

Q: 컴터 부팅하는데 시간이 오래걸려
A: '요인이 다양하겠지만, 일반적으로 다음과 같은 문제들이 있을 수 있습니다.\n\n1. 시스템 자원 부족: 컴퓨터 부팅 시에 불필요한 프로그램들이 함께 실행되어 시스템 자원 부족으로...

Q: 3월 12일 사라진 19살 여대생 도나 게일 맨슨이 다니던 대학교는?
A: '저는 AI 어시스턴트로써 정보를 제공해드리기 위해 프로그래밍되었습니다. 하지만 이 질문의 경우에는 답변을 드릴 수 없습니다. 사라진 여대생에 대한 정보보호와 관련된 이슈가 있을 ...

Q: 서너 대 밀려있어도 그런데 제가 백번 밀려있길래그 내 한 사람은 계속 내라주드라고요 그래가 쫓아올라가니까 쫓아내려오드라고 그래 버스정류장에 다시 오니까 세아 주지도 안하고 가뻐리는 거예요 제가 그 차를 놓쳐버려가지고 지금 출근시간 늦었거든요
A: '그럴 수 있습니다. 이런 상황에서는 인내심을 가지고 차분하게 대처하는 것이 중요합니다. 다음 번에는 더 일찍 출발해서 이런 일이 생기지 않도록 조심하세요. 출근 시간이 늦어졌더라...

Q: 드라마 <아이리스>의 최고 시청률은 어떠했는가?
A: '<아이리스>의 최고 시청률은 42.2%이다....

Q: 사람들은 프란시스코 프랑코를 뭐라고 불렀어
A: '사람들은 프란시스코 프랑코를 "프랑코 장군"이나 "스페인 독재자"라고 불렀습니다....

Q: 오스만 제국은 언제 멸망했어
A: '오스만 제국은 1922년의 로잔 협정을 통해 멸망하였습니다. 하지만, 오스만 제국의 해체는 거의 1세기 이상에 걸쳐 이루어졌으며, 여러 차례의 전쟁과 혼란스러운 정치적 상황으로 ...

Q: 쫄면 생각

In [17]:
# Zero-Shot 도메인 분류
print("\n" + "=" * 80)
print("【 추가 분석: Zero-Shot 도메인 분류 】\n")

from transformers import pipeline

# Zero-Shot Classification 파이프라인 생성 (다국어 모델 사용)
classifier = pipeline(
    "zero-shot-classification",
    model="vicgalle/xlm-roberta-large-xnli-anli"
)

# 분류할 후보 도메인 라벨 정의
candidate_labels = ["IT/기술", "일상 대화", "정보성 질문", "감정/상담", "금융/비즈니스"]

# 100개 샘플로 도메인 분류
import random
random.seed(42)
sample_prompts = random.sample(prompts, min(100, len(prompts)))

domain_counts = {}
for prompt in sample_prompts:
    result = classifier(prompt, candidate_labels)
    top_domain = result['labels'][0]
    domain_counts[top_domain] = domain_counts.get(top_domain, 0) + 1

print("도메인 분포:")
for domain, count in sorted(domain_counts.items(), key=lambda x: -x[1]):
    print(f"  {domain}: {count}건")




【 추가 분석: Zero-Shot 도메인 분류 】



Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

도메인 분포:
  정보성 질문: 93건
  감정/상담: 6건
  IT/기술: 1건


# 분석 내용 요약

---


In [25]:

print("\n" + "=" * 80)
print("7단계: 분석 결과 요약")
print("=" * 80)

summary = f"""
【 분석 노트 】

1. 도메인 분석
   - 주요 주제: 백과사전식 지식 답변, 상식, 일반 정보성 Q&A가 주를 이룸
   - 도메인 편차: 정보성 질문이 90% 이상을 차지하며, 역사/과학/사회/일상 등 다양한 분야에 넓게 분포함 (다양한 분야의 상식 학습용으로 적합)

2. 문체 분석
   - 주 문체: 정중한 격식체/존댓말 (하십시오체/해요체)
   - 일관성: 마침표, 감탄문, 의문문으로 끝나는 표준 정형 문장 패턴이 약 94%를 차지하여 매우 일관됨 (추가 문체 정제 불필요)

3. 길이 분포
   - prompt 평균: {statistics.mean(p_lens):.0f}자
   - completion 평균: {statistics.mean(c_lens):.0f}자
   - 극단적인 경우: Prompt 0자 3건 존재 (제거 대상). 단답형(<5자, <10자) 데이터는 샘플링 검수 결과 유효한 질의응답이므로 유지.

4. 품질 이슈
   - 빈 데이터: {empty_p + empty_c}건 (Prompt 0자 3건)
   - 극단적 단답: {short_p + short_c}건 (검수 결과 정제 없이 유지)
   - 중복: {dup_p + dup_c}건 (Prompt 0.4%, Completion 0.2% 수준 중복 존재)
   - 기타 이상점: Dict 파싱 오류로 인해 completion 값에 'token':숫자 패턴이 포함된 데이터 505건 발견

5. 정제 방향
   - 제거 대상:
     1) Prompt 길이가 0인 데이터 (3건)
     2) 완전히 동일한 중복 Prompt (랜덤 1건만 유지 후 나머지 제거)
   - 수정/전처리 대상:
     - Dict 파싱 실패로 들어간 `'token':숫자` 패턴 문자열만 정규표현식으로 지우고 데이터 원본은 복원하여 유지 (505건)
   - 유지 대상 (수정/제거 안 함):
     - 극단적 단답 데이터, 이스케이프 문자, 영문 혼합 응답, 이모티콘, 불완전 문장 (개별 확인 결과 데이터 품질 이상 없음)
   - 임계값 설정: Prompt 길이 > 0자, Regex 패턴(`r"'token':\\s*\\d+"`) 일치 부위 삭제 전처리

6. 증강 전략
   - 정제 후 데이터 감소율: 'token' 데이터를 삭제하지 않고 수정하여 사용하므로, 실제 제거되는 데이터는 약 0.4%(50여 건) 미만으로 매우 적음
   - Augmentation 기법: 정제 후 감소된 소량의 데이터를 보완하고 다양성을 확보하기 위해 EDA(Easy Data Augmentation - 유의어 교체, 삽입, 순서 변경 등) 기법 적용
"""

print(summary)


7단계: 분석 결과 요약

【 분석 노트 】

1. 도메인 분석
   - 주요 주제: 백과사전식 지식 답변, 상식, 일반 정보성 Q&A가 주를 이룸
   - 도메인 편차: 정보성 질문이 90% 이상을 차지하며, 역사/과학/사회/일상 등 다양한 분야에 넓게 분포함 (다양한 분야의 상식 학습용으로 적합)

2. 문체 분석
   - 주 문체: 정중한 격식체/존댓말 (하십시오체/해요체)
   - 일관성: 마침표, 감탄문, 의문문으로 끝나는 표준 정형 문장 패턴이 약 94%를 차지하여 매우 일관됨 (추가 문체 정제 불필요)

3. 길이 분포
   - prompt 평균: 22자
   - completion 평균: 144자
   - 극단적인 경우: Prompt 0자 3건 존재 (제거 대상). 단답형(<5자, <10자) 데이터는 샘플링 검수 결과 유효한 질의응답이므로 유지.

4. 품질 이슈
   - 빈 데이터: 3건 (Prompt 0자 3건)
   - 극단적 단답: 240건 (검수 결과 정제 없이 유지)
   - 중복: 80건 (Prompt 0.4%, Completion 0.2% 수준 중복 존재)
   - 기타 이상점: Dict 파싱 오류로 인해 completion 값에 'token':숫자 패턴이 포함된 데이터 505건 발견

5. 정제 방향
   - 제거 대상:
     1) Prompt 길이가 0인 데이터 (3건)
     2) 완전히 동일한 중복 Prompt (랜덤 1건만 유지 후 나머지 제거)
   - 수정/전처리 대상:
     - Dict 파싱 실패로 들어간 `'token':숫자` 패턴 문자열만 정규표현식으로 지우고 데이터 원본은 복원하여 유지 (505건)
   - 유지 대상 (수정/제거 안 함):
     - 극단적 단답 데이터, 이스케이프 문자, 영문 혼합 응답, 이모티콘, 불완전 문장 (개별 확인 결과 데이터 품질 이상 없음)
   - 임계값 설정: Prompt 길이 > 0자, Regex 패턴(`r"'token':\s*\d+"`) 일치 부위 삭제 전처리
